# 🏭 محرك الصيانة الوقائية وتشخيص الأعطال الصناعية

---

# المرحلة 2: استكشاف البيانات وفهم الحساسات (EDA)

بعد أن **عرّفنا المشكلة والبيانات** في المرحلة 1، ننتقل الآن لمرحلة **الاستكشاف**. الهدف ليس التزيين، بل **اتخاذ قرارات هندسية مبنية على أدلة**:

- ما توزيع كل حساس؟ وهل فيه قيم شاذة (Outliers)؟
- كيف تختلف القراءات بين الآلات **السليمة** و**المعطلة**؟ (هذا هو الفصل الذي سيتعلمه النموذج)
- أي الحساسات **أقوى ارتباطاً** بحدوث العطل؟ (سيقود هندسة الميزات لاحقاً)
- ما علاقة **جودة المنتج (Type)** بمعدل الأعطال؟

## 🎨 أدوات التصور المستخدمة

| المكتبة | الاستخدام |
|---------|-----------|
| `matplotlib` | الأساس للرسم البرمجي في بايثون |
| `seaborn` | طبقة أنيقة فوق matplotlib بتصورات إحصائية جاهزة (histplot, boxplot, heatmap) |

**لماذا نهتم بجودة التصور؟** لأن الاستكشاف البصري يكشف أنماطاً لا تظهر في الأرقام المجردة.


## 2.0 — الإعداد: الاستيراد + إعادة بناء البيانات (اختصار من المرحلة 1)

لكي يكون هذا الدفتر **قائماً بذاته** (Self-contained)، نعيد التحميل وبناء الأهداف بنسخة مختصرة من المرحلة 1. في مشروعك النهائي ستوحّد هذا المنطق في ملف `utils.py` واحد (سنفعل ذلك في المرحلة 6).


In [ ]:
# ============================================================
# المرحلة 2 — الإعداد
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ─── ثوابت (نفس المرحلة 1) ────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_FILE = Path("/kaggle/input/ai4i-predictive-maintenance-dataset/ai4i2020.csv")

SENSOR_COLUMNS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
BINARY_TARGET = "Machine failure"
FAILURE_TYPE_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
MULTI_CLASS_TARGET = "Failure Type"
CATEGORICAL_COL = "Type"   # جودة المنتج: Low / Medium / High

# ─── إعدادات التصور (نمط موحّد ونظيف) ─────────────────────
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110          # وضوح أعلى للصور
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

# ─── تحميل البيانات ────────────────────────────────────────
df = pd.read_csv(DATA_FILE)

# ─── بناء الهدف متعدد الفئات (نفس سياسة المرحلة 1) ────────
flag_sum = df[FAILURE_TYPE_COLUMNS].sum(axis=1)
df[MULTI_CLASS_TARGET] = "No Failure"
failed_mask = df[BINARY_TARGET] == 1
df.loc[failed_mask, MULTI_CLASS_TARGET] = (
    df.loc[failed_mask, FAILURE_TYPE_COLUMNS]
    .apply(lambda row: "+".join(row.index[row == 1]) or "Unknown", axis=1)
)

print(f"✅ البيانات جاهزة: {df.shape[0]:,} سجل × {df.shape[1]} عمود")
print(f"نسبة الأعطال: {df[BINARY_TARGET].mean() * 100:.2f}%")


## 2.1 — توزيع كل حساس (Histogram + KDE)

**ماذا ولماذا؟**

نرسم توزيع كل حساس لنجيب على أسئلة جوهرية:
- هل التوزيع **طبيعي/منحرف**؟ (يؤثر على اختيار مقاييس الملء والتعامل مع القيم الشاذة)
- هل توجد **قيم شاذة** أو تجمّعات غريبة؟

نستخدم `histplot` مع منحنى الكثافة `kde=True` لأنه يعطينا شكل التوزيع بسلاسة أكثر من الأعمدة وحدها.


In [ ]:
def plot_distributions(df: pd.DataFrame, cols: list[str]) -> None:
    """
    رسم توزيع كل حساس (Histogram + KDE) في شبكة فرعية.

    Parameters
    ----------
    df : pd.DataFrame
        البيانات.
    cols : list[str]
        أسماء أعمدة الحساسات الرقمية.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()

    for ax, col in zip(axes, cols):
        sns.histplot(df[col], kde=True, ax=ax, color="#4C72B0")
        ax.set_title(col, fontsize=11)
        ax.set_xlabel("")
    # إخفاء المحور السادس الفارغ
    axes[-1].set_visible(False)

    fig.suptitle("توزيع قراءات الحساسات", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

plot_distributions(df, SENSOR_COLUMNS)


## 2.2 — مقارنة القراءات بين السليمة والمعطلة (Boxplot)

**ماذا ولماذا؟**

هذه هي الرؤية **الأهم** في المسألة: نضع قراءات الحساسات جنباً إلى جنب بين فئتين:
- `0` (سليمة) و `1` (معطلة).

الـ Boxplot يكشف **الوسيط والمدى والشذوذ** لكل فئة. إذا كان توزيع فئة المعطلين **منزاحاً بوضوح** عن السليمين، فهذا الحساس **يميّز** العطل — أي أنه سيكون ميزة قوية للنموذج.


In [ ]:
def plot_target_comparison(df: pd.DataFrame, cols: list[str]) -> None:
    """
    مقارنة Boxplot لكل حساس بين الآلات السليمة والمعطلة.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()

    for ax, col in zip(axes, cols):
        sns.boxplot(data=df, x=BINARY_TARGET, y=col, ax=ax,
                    hue=BINARY_TARGET, palette=["#4C72B0", "#C44E52"], legend=False)
        ax.set_title(col, fontsize=11)
        ax.set_xlabel("Machine failure (0 = سليمة, 1 = معطلة)")

    axes[-1].set_visible(False)
    fig.suptitle("قراءات الحساسات: سليمة مقابل معطلة", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

plot_target_comparison(df, SENSOR_COLUMNS)


## 2.3 — مصفوفة الارتباط (Correlation Heatmap)

**ماذا ولماذا؟**

الارتباط يقيس **قوة واتجاه** العلاقة الخطية بين متغيرين (من -1 إلى +1):
- قريب من **+1**: كلما زاد الأول زاد الثاني.
- قريب من **-1**: علاقة عكسية.
- قريب من **0**: لا علاقة خطية.

نضيف الهدف `Machine failure` إلى المصفوفة لنرى أي الحساسات **أقرب ارتباطاً** بحدوث العطل.

> ⚠️ تنبيه منهجي: الارتباط يقيس العلاقات **الخطية فقط**. قد توجد علاقات غير خطية مهمة لا تظهر هنا — لذلك لا نبني قراراتنا على هذه المصفوفة وحدها.


In [ ]:
def plot_correlation(df: pd.DataFrame) -> None:
    """
    رسم مصفوفة الارتباط للحساسات + الهدف الثنائي.
    """
    corr_cols = SENSOR_COLUMNS + [BINARY_TARGET]
    corr = df[corr_cols].corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, vmin=-1, vmax=1, ax=ax,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    ax.set_title("مصفوفة الارتباط: الحساسات × حدوث العطل", fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()

plot_correlation(df)


## 2.4 — المتغير الفئوي (Type): جودة المنتج

**ماذا ولماذا؟**

`Type` يعبّر عن جودة المنتج: `L` (منخفضة)، `M` (متوسطة)، `H` (عالية). هذا متغير فئوي مهم لأن جودة المكوّنات تؤثر فيزيائياً على مقاومة الآلة للأعطال.

نفحص:
1. **توزيع الفئات** — هل هي متوازنة؟
2. **معدل العطل داخل كل فئة** — أي فئة أكثر عرضة للأعطال؟

هذا سيؤكد لنا أن `Type` ميزة يجب **ترميزها** (وليس حذفها) في المرحلة 3.


In [ ]:
def analyze_type(df: pd.DataFrame) -> None:
    """
    تحليل المتغير الفئوي 'Type' وعلاقته بمعدل العطل.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # (1) توزيع الفئات
    counts = df[CATEGORICAL_COL].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette="viridis")
    axes[0].set_title("توزيع جودة المنتج (Type)")
    axes[0].set_xlabel("الجودة")
    axes[0].set_ylabel("عدد السجلات")

    # (2) معدل العطل داخل كل فئة
    failure_rate = df.groupby(CATEGORICAL_COL)[BINARY_TARGET].mean() * 100
    sns.barplot(x=failure_rate.index, y=failure_rate.values, ax=axes[1], palette="rocket")
    axes[1].set_title("نسبة الأعطال (%) لكل فئة جودة")
    axes[1].set_xlabel("الجودة")
    axes[1].set_ylabel("نسبة الأعطال %")
    for i, v in enumerate(failure_rate.values):
        axes[1].text(i, v + 0.05, f"{v:.2f}%", ha="center", fontweight="bold")

    fig.tight_layout()
    plt.show()

    print("معدل العطل لكل فئة:")
    print((df.groupby(CATEGORICAL_COL)[BINARY_TARGET].mean() * 100).round(2).to_string())

analyze_type(df)


## 2.5 — علاقة كل نوع عطل بالحساسات (تحليل فزيائي)

**ماذا ولماذا؟**

هنا نربط النقاط مع **الفيزياء الحقيقية** للأعطال. من ورقة البيانات الأصلية نعرف القواعد التوليدية:

| نوع العطل | القاعدة الفيزيائية المولِّدة |
|-----------|------------------------------|
| **TWF** (تآكل الأداة) | تآكل الأداة يتجاوز عتبة (200–240 دقيقة) |
| **HDF** (تبديد الحرارة) | فرق الحرارة < 8.6K **و** السرعة < 1380 rpm |
| **PWF** (القدرة) | القدرة = العزم × السرعة < 3500W أو > 9000W |
| **OSF** (الإجهاد) | التآكل × العزم > 11,000 min·Nm |
| **RNF** (عشوائي) | عشوائي تماماً |

نرسم **متوسط تآكل الأداة** و**متوسط فرق الحرارة** لكل نوع عطل، لنتحقق بصرياً أن بياناتنا تتبع هذه القواعد — وهذا يعزز ثقتنا في صحة فهمنا للبيانات قبل النمذجة.


In [ ]:
def analyze_failure_physics(df: pd.DataFrame) -> None:
    """
    فحص اتساق البيانات مع القواعد الفيزيائية لأنواع الأعطال.
    """
    # ميزتان فيزيائيتان نفحصهما مقابل أنواع الأعطال
    df = df.copy()
    df["Temp Diff [K]"] = df["Process temperature [K]"] - df["Air temperature [K]"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # (1) متوسط تآكل الأداة لكل نوع عطل
    wear_by_type = df.groupby(MULTI_CLASS_TARGET)["Tool wear [min]"].mean()
    sns.barplot(x=wear_by_type.index, y=wear_by_type.values, ax=axes[0], palette="mako")
    axes[0].set_title("متوسط تآكل الأداة لكل نوع عطل")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].set_ylabel("Tool wear [min]")

    # (2) متوسط فرق الحرارة لكل نوع عطل
    diff_by_type = df.groupby(MULTI_CLASS_TARGET)["Temp Diff [K]"].mean()
    sns.barplot(x=diff_by_type.index, y=diff_by_type.values, ax=axes[1], palette="flare")
    axes[1].set_title("متوسط فرق الحرارة (عملية - محيط) لكل نوع عطل")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].set_ylabel("Temp Diff [K]")

    fig.tight_layout()
    plt.show()

    print("متوسط تآكل الأداة حسب النوع:")
    print(df.groupby(MULTI_CLASS_TARGET)["Tool wear [min]"].mean().round(1).to_string())

analyze_failure_physics(df)


## ✅ خلاصة المرحلة 2 (نتائج يجب تذكّرها قبل النمذجة)

من الاستكشاف نخرج بـ **رؤى هندسية** ستترجم إلى قرارات في المرحلة 3:

1. **اختلال توازن حاد (3.39%)** → سنحتاج `Stratified K-Fold` + `scale_pos_weight` وتقييم بـ Recall/F1.
2. **`Tool wear` و`Torque`** (ومشتقاتهما الفيزيائية) هما الأقوى تمييزاً للعطل → سنبني عليهما ميزات هندسية جديدة.
3. **`Type` (الجودة)** يؤثر بوضوح على معدل العطل → سيُرمَّز بـ Ordinal (لأن L < M < H ترتيب منطقي).
4. **القواعد الفيزيائية للأعطال** مطابقة للبيانات → يمكننا هندسة ميزات مثل `Power = Torque × Speed` و `Temp Diff` بثقة.

### 🔜 التالي: المرحلة 3 — المعالجة وهندسة الميزات + Pipeline
سنبني `ColumnTransformer` + `Pipeline`، ننشئ الميزات الهندسية، ونعالج عدم التوازن.
